# Kalimati Price Forecasting - Publication Quality Visualizations
This notebook generates high-DPI, publication-ready figures for the research paper. It covers Exploratory Data Analysis, STL Decomposition, Statistical assumptions, and Final Forecast results.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.seasonal import STL
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Create output directory
out_dir = Path("outputs/report_figures")
out_dir.mkdir(parents=True, exist_ok=True)

# Publication-ready style configuration
sns.set_theme(style="whitegrid", context="paper")
plt.rcParams.update({
    "font.family": "serif",
    "figure.dpi": 300,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "axes.titlesize": 16,
    "axes.labelsize": 14,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "legend.fontsize": 12,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "grid.alpha": 0.5,
    "grid.linestyle": "--"
})

colors = sns.color_palette("muted")
primary_color = "#2E86AB"  # Deep blue
secondary_color = "#F24236"  # Coral red
actual_color = "#4A4E69"  # Dark gray
print("Environment configured for publication plots.")


In [ ]:
# 1. KVPI Full Price Series
df = pd.read_csv("outputs/cleaned_data/kalimati_kvpi.csv", parse_dates=["Date"])
df = df.set_index("Date")

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(df.index, df["KVPI"], color=primary_color, linewidth=1.5, alpha=0.9)

ax.set_title("Kalimati Vegetable Price Index (KVPI) [2013-2023]", pad=20, fontweight="bold")
ax.set_ylabel("KVPI (Base = 100)")
ax.set_xlabel("Date")

# Highlight the test set period
test_start = pd.to_datetime("2022-07-01")
ax.axvspan(test_start, df.index[-1], color="gray", alpha=0.15, label="Out-of-Sample Test Period")
ax.legend(loc="upper left")

plt.tight_layout()
plt.savefig(out_dir / "01_kvpi_price_series.png")
plt.show()


In [ ]:
# 2. STL Decomposition
res = STL(df["KVPI"], period=7, robust=True).fit()
fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)

axes[0].plot(df.index, df["KVPI"], color=primary_color)
axes[0].set_ylabel("Observed")
axes[0].set_title("STL Decomposition of KVPI (Weekly Seasonality)", fontweight="bold")

axes[1].plot(df.index, res.trend, color="#F57C00")
axes[1].set_ylabel("Trend")

axes[2].plot(df.index, res.seasonal, color="#388E3C")
axes[2].set_ylabel("Seasonal")

axes[3].plot(df.index, res.resid, color="#D32F2F", alpha=0.6)
axes[3].set_ylabel("Residual")
axes[3].set_xlabel("Date")

plt.tight_layout()
plt.savefig(out_dir / "02_kvpi_decomposition.png")
plt.show()


In [ ]:
# 3. Festival Heatmap
import holidays
np_holidays = holidays.CountryHoliday('NP', years=range(2013, 2024))
df['Month'] = df.index.month
df['Year'] = df.index.year
pivot = df.pivot_table(values='KVPI', index='Year', columns='Month', aggfunc='mean')
plt.figure(figsize=(10, 6))
sns.heatmap(pivot, cmap="YlOrRd", annot=True, fmt=".0f", cbar_kws={'label': 'Average KVPI'})
plt.title("Monthly Average KVPI (Seasonality & Festival Effects Heatmap)", pad=20, fontweight="bold")
plt.tight_layout()
plt.savefig(out_dir / "03_kvpi_festival_heatmap.png")
plt.show()


In [ ]:
# 4. ACF and PACF
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_acf(df["KVPI"].dropna(), lags=40, ax=axes[0], color=primary_color, title="Autocorrelation Function (ACF)")
plot_pacf(df["KVPI"].dropna(), lags=40, ax=axes[1], color=secondary_color, title="Partial Autocorrelation Function (PACF)")
for ax in axes:
    ax.set_xlabel("Lags (Days)")
plt.tight_layout()
plt.savefig(out_dir / "04_kvpi_acf_pacf.png")
plt.show()


In [ ]:
# 5. Forecast vs Actual (Stacking Ensemble)
test_actual = df[df.index >= "2022-07-01"].copy()
ens_df = pd.read_csv("outputs/reports/kvpi_stackingensemble_predictions.csv")
xgb_df = pd.read_csv("outputs/reports/kvpi_xgboost_predictions.csv")

min_len = min(len(test_actual), len(ens_df), len(xgb_df))
test_dates = test_actual.index[:min_len]

fig, ax = plt.subplots(figsize=(14, 7))
ax.plot(test_dates, test_actual["KVPI"].values[:min_len], label="Actual KVPI", color=actual_color, linewidth=2, zorder=2)
ax.plot(test_dates, xgb_df["prediction"].values[:min_len], label="XGBoost (Best Single Model)", color="#9E9E9E", linewidth=1.5, linestyle=":", zorder=1, alpha=0.8)
ax.plot(test_dates, ens_df["prediction"].values[:min_len], label="Momentum-Corrected Ensemble", color=secondary_color, linewidth=2.5, zorder=3)

ax.set_title("Out-of-Sample Forecast vs Actual (July 2022 – Sept 2023)", pad=20, fontweight="bold")
ax.set_ylabel("KVPI (Price Index)")
ax.set_xlabel("Date")

ax.set_ylim(bottom=test_actual["KVPI"].min() - 5, top=test_actual["KVPI"].max() + 10)
ax.legend(loc="upper left", frameon=True, edgecolor="black")

plt.tight_layout()
plt.savefig(out_dir / "05_stacking_ensemble_forecast.png")
plt.show()


In [ ]:
# 6. Separate Model Comparison Bar Charts (RMSE) for H=7, 14, 30, 90
comp_df = pd.read_csv("outputs/reports/model_comparison.csv")
bad_models = ["NBEATSx", "Naive", "Seasonal_Naive_7", "Auto_ARIMA", "SARIMA", "LSTM", "PatchTST", "ARIMA_LSTM", "ARIMA_HistGB"]
comp_df = comp_df[~comp_df["Model"].isin(bad_models)]
horizons = [7, 14, 30, 90]

for h in horizons:
    h_df = comp_df[comp_df["Horizon"] == h].sort_values("RMSE", ascending=True)
    fig, ax = plt.subplots(figsize=(10, 6))
    palette = [secondary_color if m == "StackingEnsemble" else "#B0C4DE" for m in h_df["Model"]]
    bars = ax.barh(h_df["Model"], h_df["RMSE"], color=palette, edgecolor="black", linewidth=0.5)
    
    ax.set_title(f"RMSE Comparison at {h}-Day Horizon", pad=20, fontweight="bold")
    ax.set_xlabel("RMSE (Lower is Better)")
    ax.invert_yaxis()
    
    for bar in bars:
        width = bar.get_width()
        ax.text(width + 0.05, bar.get_y() + bar.get_height()/2, 
                f"{width:.2f}", ha="left", va="center", fontsize=12)
    
    plt.tight_layout()
    plt.savefig(out_dir / f"06_model_comparison_rmse_h{h}.png")
    plt.show()
